# 002 — Metadata Extraction
**Ingestion Series** | Enrich every chunk with structured metadata

Metadata (entities, dates, topics, section headers) dramatically improves
filtered retrieval. A query for "2023 AI regulations" needs a date filter — not
just semantic similarity.

Covers: Pydantic structured output · Entity/date/topic extraction · Metadata-enriched documents


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile
✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)


In [3]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


✓ 5 dataset files loaded
  File                    Chars   ~Words
  ----------------------------------------
  ai.txt                  6,221      943
  machine_learning.txt    5,953      870
  nlp.txt                 5,160      693
  deep_learning.txt       5,114      686
  rag.txt                 5,146      715
  ----------------------------------------
  TOTAL                  27,602    3,907


In [4]:
# ── Pydantic schema for structured extraction ────────────────────────────
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

class ChunkMetadata(BaseModel):
    entities:   List[str] = Field(description="Named entities: people, orgs, places, products")
    dates:      List[str] = Field(description="Any dates or time references mentioned")
    topics:     List[str] = Field(description="Main topics (2-5 keywords)")
    summary:    str       = Field(description="One-sentence summary of this chunk")
    section:    str       = Field(description="Inferred section/heading for this chunk")

print("✓ ChunkMetadata schema defined")
print(ChunkMetadata.model_json_schema())


✓ ChunkMetadata schema defined
{'properties': {'entities': {'description': 'Named entities: people, orgs, places, products', 'items': {'type': 'string'}, 'title': 'Entities', 'type': 'array'}, 'dates': {'description': 'Any dates or time references mentioned', 'items': {'type': 'string'}, 'title': 'Dates', 'type': 'array'}, 'topics': {'description': 'Main topics (2-5 keywords)', 'items': {'type': 'string'}, 'title': 'Topics', 'type': 'array'}, 'summary': {'description': 'One-sentence summary of this chunk', 'title': 'Summary', 'type': 'string'}, 'section': {'description': 'Inferred section/heading for this chunk', 'title': 'Section', 'type': 'string'}}, 'required': ['entities', 'dates', 'topics', 'summary', 'section'], 'title': 'ChunkMetadata', 'type': 'object'}


In [5]:
# ── LLM extraction chain ─────────────────────────────────────────────────
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

parser = JsonOutputParser(pydantic_object=ChunkMetadata)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured metadata from the text. Return valid JSON only."),
    ("human", "Text:\n{text}\n\nFormat:\n{format_instructions}"),
]).partial(format_instructions=parser.get_format_instructions())

extract_chain = prompt | llm | parser

# Test on one chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
chunks = splitter.split_text(ai_text[:6000])

sample_chunk = chunks[1]
print("Chunk text (first 300 chars):")
print(sample_chunk[:300])
print()

metadata = extract_chain.invoke({"text": sample_chunk})
print("Extracted metadata:")
import json
print(json.dumps(metadata, indent=2))


Chunk text (first 300 chars):
AI applications include advanced web search engines such as Google Search, recommendation systems
used by YouTube, Amazon, and Netflix, understanding human speech such as Siri and Alexa,
self-driving cars such as Waymo, generative AI tools such as ChatGPT, and competing at the
highest level in strat



Extracted metadata:
{
  "entities": [
    "Google Search",
    "YouTube",
    "Amazon",
    "Netflix",
    "Siri",
    "Alexa",
    "Waymo",
    "ChatGPT",
    "chess",
    "Go"
  ],
  "dates": [],
  "topics": [
    "AI applications",
    "advanced web search engines",
    "recommendation systems",
    "self-driving cars",
    "generative AI tools",
    "strategic games"
  ],
  "summary": "AI applications include advanced web search engines, recommendation systems, self-driving cars, generative AI tools, and competing at the highest level in strategic games.",
  "section": "History of Artificial Intelligence"
}


In [6]:
# ── Batch extraction over multiple chunks ────────────────────────────────
def extract_metadata_batch(chunks, max_chunks=5):
    enriched_docs = []
    for i, chunk in enumerate(chunks[:max_chunks]):
        try:
            meta = extract_chain.invoke({"text": chunk})
            doc = Document(
                page_content=chunk,
                metadata={
                    "entities":  meta.get("entities", []),
                    "dates":     meta.get("dates", []),
                    "topics":    meta.get("topics", []),
                    "summary":   meta.get("summary", ""),
                    "section":   meta.get("section", ""),
                    "chunk_id":  i,
                }
            )
        except Exception as e:
            doc = Document(page_content=chunk, metadata={"chunk_id": i, "error": str(e)})
        enriched_docs.append(doc)
        print(f"  Processed chunk {i+1}/{min(max_chunks, len(chunks))}: "
              f"{doc.metadata.get('section','?')} | topics: {doc.metadata.get('topics',[])}")
    return enriched_docs

enriched_docs = extract_metadata_batch(chunks, max_chunks=4)
print(f"\n✓ {len(enriched_docs)} metadata-enriched documents ready")


  Processed chunk 1/4: Introduction to Artificial Intelligence | topics: ['Artificial Intelligence', 'Machine Learning', 'Intelligence', 'Agents', 'Environment']


  Processed chunk 2/4: History of Artificial Intelligence | topics: ['AI applications', 'advanced web search engines', 'recommendation systems', 'self-driving cars', 'generative AI tools', 'strategic games']


  Processed chunk 3/4: History of Artificial Intelligence | topics: ['Artificial Intelligence', 'History of AI']


  Processed chunk 4/4: The first AI winter (1974–1980) | topics: ['AI winter', 'expert systems', 'neural networks']

✓ 4 metadata-enriched documents ready


In [7]:
# ── Metadata-filtered retrieval demo ─────────────────────────────────────
from langchain_community.vectorstores import FAISS

# Build vector store with metadata-rich docs
vectorstore = FAISS.from_documents(enriched_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Regular retrieval (no filter)
query = "What are the main applications of AI?"
results = retriever.invoke(query)
print(f"Query: {query!r}")
print(f"\nTop result metadata:")
print(f"  section : {results[0].metadata.get('section')}")
print(f"  topics  : {results[0].metadata.get('topics')}")
print(f"  summary : {results[0].metadata.get('summary')}")
print(f"  entities: {results[0].metadata.get('entities', [])[:3]}")
print(f"  snippet : {results[0].page_content[:200]}")


Query: 'What are the main applications of AI?'

Top result metadata:
  section : History of Artificial Intelligence
  topics  : ['AI applications', 'advanced web search engines', 'recommendation systems', 'self-driving cars', 'generative AI tools', 'strategic games']
  summary : AI applications include advanced web search engines, recommendation systems, self-driving cars, generative AI tools, and competing at the highest level in strategic games.
  entities: ['Google Search', 'YouTube', 'Amazon']
  snippet : AI applications include advanced web search engines such as Google Search, recommendation systems
used by YouTube, Amazon, and Netflix, understanding human speech such as Siri and Alexa,
self-driving 


In [8]:
# ── Keyword metadata filter ───────────────────────────────────────────────
# Simulate filtering by topic keyword
target_topic = "machine learning"

filtered = [doc for doc in enriched_docs
            if any(target_topic.lower() in t.lower()
                   for t in doc.metadata.get("topics", []))]

print(f"Docs mentioning '{target_topic}': {len(filtered)}")
for doc in filtered:
    print(f"  • {doc.metadata.get('summary', 'n/a')[:120]}")


Docs mentioning 'machine learning': 1
  • Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to natural intelligence displayed by a


## Key Takeaways
- Metadata extraction with Pydantic + Groq gives **structured, filterable** chunk data
- Entities, dates, and section headers enable **hybrid retrieval** (vector + filter)
- Extract at ingest time — cheap once, valuable forever
- For production: batch with async, cache results, store in a doc DB alongside the vector index
